In [1]:
# ---- Setup (works locally and on Google Colab) ----
import os, sys, zipfile
if 'google.colab' in sys.modules and not os.path.isdir('images'):
    from google.colab import files
    print("Upload Lab05_Submission.zip (the zip that contains this notebook and the images folder)")
    up = files.upload()
    for n in up:
        if n.endswith('.zip'):
            zipfile.ZipFile(n).extractall('.')
    if os.path.isdir('Lab05_Submission'):
        os.chdir('Lab05_Submission')

import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from collections import deque

print("OpenCV", cv2.__version__, "| NumPy", np.__version__)
IMG, OUT = 'images', 'outputs'
os.makedirs(OUT, exist_ok=True)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

def load_gray(name):
    im = cv2.imread(os.path.join(IMG, name), cv2.IMREAD_GRAYSCALE)
    assert im is not None, f"cannot read {name}"
    return im

def load_color(name):
    im = cv2.imread(os.path.join(IMG, name), cv2.IMREAD_COLOR)
    assert im is not None, f"cannot read {name}"
    return im

def rgb(bgr): return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def save(name, img): cv2.imwrite(os.path.join(OUT, name), img)

def show_row(items, save_as=None, cols=None, size=4, suptitle=None, caption=None):
    n = len(items); cols = cols or n; rows = int(np.ceil(n / cols))
    fig, axs = plt.subplots(rows, cols, figsize=(size * cols, size * rows * 0.95))
    axs = np.atleast_1d(axs).ravel()
    for ax, (t, im) in zip(axs, items):
        if im.ndim == 2: ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        else: ax.imshow(rgb(im))
        ax.set_title(t, fontsize=10); ax.axis('off')
    for ax in axs[n:]: ax.axis('off')
    if suptitle: fig.suptitle(suptitle, fontsize=13)
    plt.tight_layout()
    if caption: fig.text(0.5, -0.01, caption, ha='center', va='top', fontsize=10, wrap=True)
    if save_as: fig.savefig(os.path.join(OUT, save_as), dpi=120, bbox_inches='tight')
    plt.show()

OpenCV 5.0.0 | NumPy 2.5.3


In [2]:
gray1 = load_gray('task1.png')
print("image size:", gray1.shape)

# --- how uneven is the illumination? mean brightness of each column/row band
H, W = gray1.shape
col_mean = gray1.mean(axis=0)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(col_mean); ax[0].set_title('Mean brightness per column (left -> right)'); ax[0].set_xlabel('x'); ax[0].set_ylabel('gray')
ax[1].hist(gray1.ravel(), bins=64, color='gray'); ax[1].set_title('Grey-level histogram')
plt.tight_layout(); plt.show()
print("mean brightness left third / middle / right third:",
      [round(float(gray1[:, i*W//3:(i+1)*W//3].mean()),1) for i in range(3)])
print("mean brightness top third / middle / bottom third:",
      [round(float(gray1[i*H//3:(i+1)*H//3, :].mean()),1) for i in range(3)])

image size: (563, 558)
mean brightness left third / middle / right third: [80.8, 99.5, 125.1]
mean brightness top third / middle / bottom third: [114.2, 105.6, 85.6]


In [3]:
# --- global thresholding with 3 different values (ink = black, paper = white)
T1, T2, T3 = 70, 110, 150
g1 = cv2.threshold(gray1, T1, 255, cv2.THRESH_BINARY)[1]
g2 = cv2.threshold(gray1, T2, 255, cv2.THRESH_BINARY)[1]
g3 = cv2.threshold(gray1, T3, 255, cv2.THRESH_BINARY)[1]

# --- adaptive thresholding (Gaussian weighted neighbourhood)
BLOCK1, C1 = 15, 8
ad1 = cv2.adaptiveThreshold(gray1, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, BLOCK1, C1)

def ink_stats(mask, n=3):
    # share of ink (black) pixels inside n vertical strips -> reveals illumination bias
    return [round(float((mask[:, i*W//n:(i+1)*W//n] == 0).mean() * 100), 1) for i in range(n)]

rows = []
for name, m in [(f'Global T={T1}', g1), (f'Global T={T2}', g2), (f'Global T={T3}', g3), (f'Adaptive G b={BLOCK1},C={C1}', ad1)]:
    s = ink_stats(m)
    rows.append([name, round(float((m == 0).mean() * 100), 1), *s])
cmp1 = pd.DataFrame(rows, columns=['method', 'ink % (all)', 'ink % left', 'ink % middle', 'ink % right'])
cmp1

,method,ink % (all),ink % left,ink % middle,ink % right
0,Global T=70,18.7,35.9,13.3,7.0
1,Global T=110,57.3,87.9,60.8,23.1
2,Global T=150,96.8,98.9,100.0,91.4
3,"Adaptive G b=15,C=8",13.6,11.2,14.2,15.4


In [4]:
BEST_GLOBAL = g1   # T=70 chosen after inspecting the 3 masks (see analysis below)
show_row([('Original (gray)', gray1), (f'Global T1={T1}', g1), (f'Global T2={T2}', g2),
          (f'Global T3={T3}', g3), (f'Adaptive Gaussian\nblock={BLOCK1}, C={C1}', ad1)],
         save_as='task1_comparison.png', size=3.6,
         caption="Why does a single threshold struggle when illumination changes across the image?  Because one fixed T compares every pixel with the same grey level, "
                 "but the paper's own brightness changes across the page: a T that is right for the bright side turns the dark side into ink, "
                 "and a T that is right for the dark side loses the text on the bright side.")
save('task1_final_adaptive_mask.png', ad1)
save('task1_best_global_mask.png', BEST_GLOBAL)

In [5]:
gray2 = load_gray('task2.png')
blocks = [7, 31, 101]
Cs = [2, 10, 25]
methods = {'Mean': cv2.ADAPTIVE_THRESH_MEAN_C, 'Gaussian': cv2.ADAPTIVE_THRESH_GAUSSIAN_C}

res2, rows = {}, []
for mname, m in methods.items():
    for b in blocks:
        for C in Cs:
            mask = cv2.adaptiveThreshold(gray2, 255, m, cv2.THRESH_BINARY, b, C)
            res2[(mname, b, C)] = mask
            ink = (mask == 0).astype(np.uint8)
            n, lab, st, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)
            areas = st[1:, cv2.CC_STAT_AREA]
            rows.append([mname, b, C, round(ink.mean() * 100, 2), n - 1, int((areas <= 3).sum()), int(areas.max()) if len(areas) else 0])
tab2 = pd.DataFrame(rows, columns=['method', 'blockSize', 'C', 'ink %', '#components', '#speckles (<=3px)', 'largest comp. (px)'])
tab2.to_csv(os.path.join(OUT, 'task2_parameter_table.csv'), index=False)
tab2

,method,blockSize,C,ink %,#components,#speckles (<=3px),largest comp. (px)
0,Mean,7,2,24.18,4824,2489,26792
1,Mean,7,10,11.76,439,109,12940
2,Mean,7,25,4.68,846,463,3193
3,Mean,31,2,24.89,965,448,35603
4,Mean,31,10,17.38,193,63,28120
5,Mean,31,25,10.93,419,130,15658
6,Mean,101,2,24.52,298,151,34925
7,Mean,101,10,18.91,177,59,27535
8,Mean,101,25,12.46,323,116,16487
9,Gaussian,7,2,20.52,6139,4373,24445


In [6]:
for mname in methods:
    items = [('Original', gray2)] + []
    fig, axs = plt.subplots(len(blocks), len(Cs), figsize=(4 * len(Cs), 3.9 * len(blocks)))
    for i, b in enumerate(blocks):
        for j, C in enumerate(Cs):
            axs[i, j].imshow(res2[(mname, b, C)], cmap='gray', vmin=0, vmax=255)
            axs[i, j].set_title(f'{mname}  blockSize={b}, C={C}', fontsize=10); axs[i, j].axis('off')
    plt.tight_layout()
    fig.savefig(os.path.join(OUT, f'task2_{mname.lower()}_grid.png'), dpi=110, bbox_inches='tight'); plt.show()

In [7]:
BEST2 = ('Gaussian', 31, 10)   # chosen after inspecting the grids / table (see analysis below)
sel = [('Mean', 7, 2), ('Mean', 31, 10), ('Mean', 101, 25), ('Gaussian', 7, 2), BEST2, ('Gaussian', 101, 25)]
show_row([('Original', gray2)] + [(f'{m}  b={b}, C={c}', res2[(m, b, c)]) for m, b, c in sel],
         save_as='task2_final_comparison.png', cols=4, size=3.8)
save('task2_best_mask.png', res2[BEST2])
print("best combination:", BEST2)

best combination: ('Gaussian', 31, 10)


In [8]:
gray3 = load_gray('task3.jpg')
hist3 = cv2.calcHist([gray3], [0], None, [256], [0, 256]).ravel()

# Otsu (objects are darker than the background -> INV so that objects become white)
t_otsu, mask_otsu = cv2.threshold(gray3, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
print("Otsu threshold selected automatically:", t_otsu)

fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
ax[0].imshow(gray3, cmap='gray'); ax[0].set_title('Original (gray)'); ax[0].axis('off')
ax[1].bar(range(256), hist3, color='gray', width=1.0); ax[1].axvline(t_otsu, color='r', ls='--', label=f'Otsu T = {t_otsu:.0f}')
ax[1].set_title('Histogram'); ax[1].set_xlabel('grey level'); ax[1].legend()
ax[2].imshow(mask_otsu, cmap='gray'); ax[2].set_title('Otsu binary mask'); ax[2].axis('off')
plt.tight_layout(); fig.savefig(os.path.join(OUT, 'task3_original_hist_otsu.png'), dpi=120, bbox_inches='tight'); plt.show()
save('task3_otsu_mask.png', mask_otsu)

Otsu threshold selected automatically: 162.0


In [9]:
# --- repeat after changing contrast / pre-processing slightly
def contrast(img, alpha, beta=0):
    return np.clip(alpha * (img.astype(np.float32) - 128) + 128 + beta, 0, 255).astype(np.uint8)

variants = {
    'original': gray3,
    'contrast x0.6': contrast(gray3, 0.6),
    'contrast x1.4': contrast(gray3, 1.4),
    'brightness +40': np.clip(gray3.astype(np.int16) + 40, 0, 255).astype(np.uint8),
    'Gaussian blur 5x5': cv2.GaussianBlur(gray3, (5, 5), 0),
    'median blur 5': cv2.medianBlur(gray3, 5),
}
rows, items = [], []
for k, im in variants.items():
    t, m = cv2.threshold(im, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    n, _, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    big = int((st[1:, cv2.CC_STAT_AREA] > 100).sum())
    rows.append([k, t, round(float((m > 0).mean() * 100), 1), big]); items.append((f'{k}\nOtsu T={t:.0f}', m))
tab3 = pd.DataFrame(rows, columns=['pre-processing', 'Otsu threshold', 'foreground %', '#objects (area>100)'])
tab3.to_csv(os.path.join(OUT, 'task3_threshold_comparison.csv'), index=False)
show_row(items, save_as='task3_variants.png', cols=3, size=3.8)
tab3

,pre-processing,Otsu threshold,foreground %,#objects (area>100)
0,original,162.0,55.9,1
1,contrast x0.6,148.0,55.9,1
2,contrast x1.4,161.0,55.4,1
3,brightness +40,191.0,55.3,1
4,Gaussian blur 5x5,162.0,56.5,1
5,median blur 5,162.0,56.2,1


In [10]:
bgr4 = load_color('task4.png')
hsv4 = cv2.cvtColor(bgr4, cv2.COLOR_BGR2HSV)

# sample the HSV values of the three blue sweets (x, y) to choose sensible bounds
for (x, y) in [(375, 80), (347, 237), (293, 320)]:
    print((x, y), "HSV =", hsv4[y-2:y+3, x-2:x+3].reshape(-1, 3).mean(axis=0).round(0))
print("background HSV:", hsv4[20, 20])

(375, 80) HSV = [106. 201. 247.]
(347, 237) HSV = [112. 191. 247.]
(293, 320) HSV = [110. 193. 247.]
background HSV: [  0   0 255]


In [11]:
def hsv_mask(lower, upper):
    return cv2.inRange(hsv4, np.array(lower, np.uint8), np.array(upper, np.uint8))

# experiments with the limits
exps = {
    'too restrictive  H 108-112, S 200-255, V 200-255': ([108, 200, 200], [112, 255, 255]),
    'narrow H 100-125, S 150-255, V 100-255':           ([100, 150, 100], [125, 255, 255]),
    'FINAL  H 95-135, S 80-255, V 40-255':               ([95, 80, 40],    [135, 255, 255]),
    'too loose H 60-150, S 20-255, V 0-255':             ([60, 20, 0],     [150, 255, 255]),
}
items = [('Original', bgr4)]
rows = []
for k, (lo, up) in exps.items():
    m = hsv_mask(lo, up)
    n, _, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    rows.append([k, int((m > 0).sum()), int((st[1:, cv2.CC_STAT_AREA] > 50).sum())])
    items.append((k.replace('  ', '\n', 1), m))
show_row(items, cols=5, size=3.6, save_as='task4_comparison.png')
pd.DataFrame(rows, columns=['limits', 'mask pixels', '#blobs (>50 px)'])

,limits,mask pixels,#blobs (>50 px)
0,"too restrictive H 108-112, S 200-255, V 200-255",6,0
1,"narrow H 100-125, S 150-255, V 100-255",6054,3
2,"FINAL H 95-135, S 80-255, V 40-255",7075,4
3,"too loose H 60-150, S 20-255, V 0-255",13721,8


In [12]:
restrictive = hsv_mask([108, 200, 200], [112, 255, 255])
final_mask = hsv_mask([95, 80, 40], [135, 255, 255])
extract_final = cv2.bitwise_and(bgr4, bgr4, mask=final_mask)
extract_restr = cv2.bitwise_and(bgr4, bgr4, mask=restrictive)

show_row([('Original', bgr4), ('Restrictive mask (fails)', restrictive), ('Final mask', final_mask),
          ('Extraction with final mask', extract_final)], save_as='task4_final_extraction.png', size=4)
save('task4_restrictive_mask.png', restrictive)
save('task4_final_mask.png', final_mask)
print("restrictive covers", int((restrictive>0).sum()), "px  |  final covers", int((final_mask>0).sum()), "px")

restrictive covers 6 px  |  final covers 7075 px


In [13]:
gray5 = load_gray('task5.png')
blur5 = cv2.GaussianBlur(gray5, (5, 5), 1.4)

pairs = [(20, 60), (50, 150), (150, 300)]      # (low, high)
edges5 = [cv2.Canny(blur5, lo, hi) for lo, hi in pairs]
print("edge pixels:", {p: int((e > 0).sum()) for p, e in zip(pairs, edges5)})
show_row([('Original', gray5)] + [(f'Edge result {i+1}: low={lo}, high={hi}', e) for i, ((lo, hi), e) in enumerate(zip(pairs, edges5))],
         save_as='task5_canny_comparison.png', size=3.9)
save('task5_final_edges.png', edges5[1])

edge pixels: {(20, 60): 3533, (50, 150): 3038, (150, 300): 2619}


In [14]:
# --- effect of raising the HIGH threshold while LOW stays fixed
LOW = 50
highs = [60, 100, 150, 250, 400]
sweep = [cv2.Canny(blur5, LOW, h) for h in highs]
show_row([(f'low={LOW}, high={h}\n{int((e>0).sum())} edge px', e) for h, e in zip(highs, sweep)],
         save_as='task5_fixed_low_sweep.png', size=3.3)

In [15]:
# --- strong vs weak edges for the middle pair. Gradient magnitude of the blurred image (Sobel, L1 norm as used by Canny)
lo, hi = pairs[1]
gx = cv2.Sobel(blur5, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(blur5, cv2.CV_32F, 0, 1, ksize=3)
mag = np.abs(gx) + np.abs(gy)
e = edges5[1] > 0
strong = e & (mag >= hi)          # kept because they exceed the high threshold
weak = e & (mag < hi)             # kept only because hysteresis links them to a strong edge
vis = cv2.cvtColor(gray5, cv2.COLOR_GRAY2BGR) // 3
vis[strong] = (0, 0, 255); vis[weak] = (255, 160, 0)
# edges that exist with the permissive pair but disappear with the strict one
permissive, strict = edges5[0] > 0, edges5[2] > 0
lost = cv2.dilate(permissive.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
lost = permissive & ~cv2.dilate(strict.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
vis2 = cv2.cvtColor(gray5, cv2.COLOR_GRAY2BGR) // 3
vis2[strict] = (0, 0, 255); vis2[lost] = (0, 200, 0)
show_row([('Pair 2: strong (red) / weak-kept (blue)', vis), ('strict pair 3 edges (red) vs edges it misses (green)', vis2)],
         save_as='task5_strong_weak.png', size=5.5)
print("strong edge px:", int(strong.sum()), "| weak (hysteresis-kept) px:", int(weak.sum()), "| edges lost by strict pair:", int(lost.sum()))

strong edge px: 2860 | weak (hysteresis-kept) px: 178 | edges lost by strict pair: 894


In [16]:
gray6 = load_gray('task6.png')
smooth6 = cv2.GaussianBlur(gray6, (5, 5), 0)          # light smoothing: region growing is very sensitive to single noisy pixels

def region_grow(img, seed, thr, connectivity=8):
    """seed = (row, col). Returns a binary mask (0/255). Compares each candidate pixel with the mean of a 5x5 patch around the seed."""
    h, w = img.shape
    r0, c0 = seed
    ref = float(img[max(r0-2, 0):r0+3, max(c0-2, 0):c0+3].mean())
    mask = np.zeros((h, w), np.uint8)
    nbrs = [(-1,0),(1,0),(0,-1),(0,1)] + ([(-1,-1),(-1,1),(1,-1),(1,1)] if connectivity == 8 else [])
    q = deque([seed]); mask[r0, c0] = 255
    while q:
        r, c = q.popleft()
        for dr, dc in nbrs:
            rr, cc = r + dr, c + dc
            if 0 <= rr < h and 0 <= cc < w and mask[rr, cc] == 0 and abs(float(img[rr, cc]) - ref) <= thr:
                mask[rr, cc] = 255; q.append((rr, cc))
    return mask, ref

seeds = {'A (bright lesion)': (265, 165), 'B (normal tissue, right hemisphere)': (200, 350)}
thrs = [15, 30, 50]
for k, s in seeds.items(): print(k, "seed (row, col) =", s, "intensity =", int(smooth6[s]))

A (bright lesion) seed (row, col) = (265, 165) intensity = 208
B (normal tissue, right hemisphere) seed (row, col) = (200, 350) intensity = 131


In [17]:
rows, results6 = [], {}
fig, axs = plt.subplots(len(seeds), len(thrs) + 1, figsize=(4.2 * (len(thrs) + 1), 3.9 * len(seeds)))
for i, (sname, s) in enumerate(seeds.items()):
    axs[i, 0].imshow(gray6, cmap='gray'); axs[i, 0].plot(s[1], s[0], 'r+', ms=14, mew=2); axs[i, 0].set_title(f'Seed {sname}'); axs[i, 0].axis('off')
    for j, t in enumerate(thrs):
        m, ref = region_grow(smooth6, s, t)
        results6[(sname, t)] = m
        ov = cv2.cvtColor(gray6, cv2.COLOR_GRAY2BGR); ov[m > 0] = (0.4 * ov[m > 0] + 0.6 * np.array([0, 0, 255])).astype(np.uint8)
        axs[i, j + 1].imshow(rgb(ov)); axs[i, j + 1].set_title(f'thr={t}  ({int((m>0).sum())} px, {100*(m>0).mean():.1f}%)', fontsize=10); axs[i, j + 1].axis('off')
        rows.append([sname, s, round(ref, 1), t, int((m > 0).sum()), round(100 * (m > 0).mean(), 2)])
plt.tight_layout(); fig.savefig(os.path.join(OUT, 'task6_region_growing.png'), dpi=110, bbox_inches='tight'); plt.show()
tab6 = pd.DataFrame(rows, columns=['seed', '(row, col)', 'seed intensity', 'threshold', 'region px', 'image %'])
tab6.to_csv(os.path.join(OUT, 'task6_parameter_table.csv'), index=False)
tab6

,seed,"(row, col)",seed intensity,threshold,region px,image %
0,A (bright lesion),"(265, 165)",208.1,15,3612,1.63
1,A (bright lesion),"(265, 165)",208.1,30,5079,2.29
2,A (bright lesion),"(265, 165)",208.1,50,6332,2.86
3,"B (normal tissue, right hemisphere)","(200, 350)",129.8,15,33756,15.24
4,"B (normal tissue, right hemisphere)","(200, 350)",129.8,30,55236,24.93
5,"B (normal tissue, right hemisphere)","(200, 350)",129.8,50,67580,30.50


In [18]:
FINAL6 = ('A (bright lesion)', 50)
final6 = results6[FINAL6]
show_row([('Original MRI', gray6), (f'Binary mask  seed={seeds[FINAL6[0]]}, thr={FINAL6[1]}', final6)], save_as='task6_final_mask_view.png', size=5)
save('task6_final_mask.png', final6)

In [19]:
def watershed_pipeline(bgr, dist_ratio=0.5, blur=(5, 5), open_iter=2, bg_iter=3, kernel=3):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, blur, 0)                                         # 1 pre-processing
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)  # 2 thresholding
    k = np.ones((kernel, kernel), np.uint8)
    opened = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, k, iterations=open_iter)       # 3 noise removal
    sure_bg = cv2.dilate(opened, k, iterations=bg_iter)                               # 4 sure background
    dist = cv2.distanceTransform(opened, cv2.DIST_L2, 5)                              # 5 distance transform
    dist_thr = dist_ratio * dist.max()
    _, sure_fg = cv2.threshold(dist, dist_thr, 255, 0)                                # 6 sure foreground
    sure_fg = np.uint8(sure_fg)
    unknown = cv2.subtract(sure_bg, sure_fg)                                          # 7 unknown region
    n_markers, markers = cv2.connectedComponents(sure_fg)                             # 8 marker labelling
    markers = markers + 1                      # background = 1, coins = 2,3,...
    markers[unknown == 255] = 0                # unknown = 0 (to be decided by watershed)
    markers = cv2.watershed(bgr.copy(), markers)                                      # 9 watershed
    out = bgr.copy(); out[markers == -1] = (0, 0, 255)                                # 10 boundaries in red
    return dict(gray=blurred, thresh=thresh, opened=opened, sure_bg=sure_bg, dist=dist, dist_thr=dist_thr,
                sure_fg=sure_fg, unknown=unknown, n_fg=n_markers - 1, markers=markers, out=out)

bgr7 = load_color('task7.jpg')
P7 = dict(dist_ratio=0.5, blur=(5, 5), open_iter=2, bg_iter=3, kernel=3)
r7 = watershed_pipeline(bgr7, **P7)
dist_vis = cv2.normalize(r7['dist'], None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
n_regions7 = len([l for l in np.unique(r7['markers']) if l > 1])
print("parameters:", P7, "| distance threshold =", round(float(r7['dist_thr']), 2), "(max dist =", round(float(r7['dist'].max()), 2), ")")
print("foreground markers:", r7['n_fg'], "| separated regions after watershed:", n_regions7)
show_row([('Original', bgr7), ('Threshold (Otsu inv.)', r7['thresh']), ('Sure background', r7['sure_bg']), ('Distance transform', dist_vis),
          ('Sure foreground', r7['sure_fg']), ('Unknown region', r7['unknown']), (f'Final watershed ({n_regions7} regions)', r7['out'])],
         save_as='task7_watershed_pipeline.png', cols=4, size=3.8)
save('task7_final_watershed.png', r7['out'])

parameters: {'dist_ratio': 0.5, 'blur': (5, 5), 'open_iter': 2, 'bg_iter': 3, 'kernel': 3} | distance threshold = 12.0 (max dist = 24.0 )
foreground markers: 24 | separated regions after watershed: 24


In [20]:
bgr8 = load_color('task8.jpg')
ratios = [0.3, 0.4, 0.6, 0.8, 0.95]
rows, items, outs8 = [], [], {}
for rt in ratios:
    r = watershed_pipeline(bgr8, dist_ratio=rt)
    mk = r['markers']
    labels = [l for l in np.unique(mk) if l > 1]
    areas = np.array([(mk == l).sum() for l in labels])
    med = np.median(areas)
    merged = int((areas > 1.5 * med).sum()); tiny = int((areas < 0.5 * med).sum())
    cv_area = float(areas.std() / areas.mean())      # all coins have the same size -> a good result has a very small spread
    rows.append([len(rows) + 1, rt, round(float(r['dist_thr']), 2), r['n_fg'], len(labels), merged, tiny, round(cv_area, 3)])
    outs8[rt] = r['out']; items.append((f'ratio={rt}  thr={r["dist_thr"]:.1f}\n{r["n_fg"]} markers / {len(labels)} regions', r['out']))
tab8 = pd.DataFrame(rows, columns=['Experiment', 'Distance ratio', 'Distance threshold', 'Foreground markers', 'Separated regions',
                                   'oversized regions (>1.5x median)', 'tiny regions (<0.5x median)', 'region-area spread (std/mean)'])
show_row(items, save_as='task8_tuning.png', cols=3, size=4.6)
tab8

,Experiment,Distance ratio,Distance threshold,Foreground markers,Separated regions,oversized regions (>1.5x median),tiny regions (<0.5x median),region-area spread (std/mean)
0,1,0.30,7.2,1,1,0,0,0.000
1,2,0.40,9.6,7,7,2,1,1.049
2,3,0.60,14.4,24,24,0,0,0.042
3,4,0.80,19.2,24,24,0,0,0.037
4,5,0.95,22.8,24,24,3,3,0.510


In [21]:
# visual judgement written after inspecting each result in the figure above
observed8 = ['All coins fuse into ONE marker (the sure foreground still contains the necks between touching coins); count = 1, completely wrong.', 'Only 7 markers for 24 coins: chains of touching coins share a marker, 2 oversized regions and 1 tiny fragment (area spread 1.05). Merged coins.', '24 markers / 24 regions, area spread 0.042: every coin has its own region and the boundaries run along the necks between touching coins. Best.', '24 markers / 24 regions, area spread 0.037: still correct; the markers are small cores but the watershed still finds the right boundaries.', '24 markers but markers are tiny cores: 3 oversized and 3 tiny regions (area spread 0.51), so some coins get a sliver while a neighbour absorbs its area. Merged and split errors.']
tab8['Observed result'] = observed8
tab8.to_csv(os.path.join(OUT, 'task8_table.csv'), index=False)
BEST8 = 0.6
save('task8_best_watershed.png', outs8[BEST8])
print("best ratio:", BEST8)
tab8

best ratio: 0.6


,Experiment,Distance ratio,Distance threshold,Foreground markers,Separated regions,oversized regions (>1.5x median),tiny regions (<0.5x median),region-area spread (std/mean),Observed result
0,1,0.30,7.2,1,1,0,0,0.000,All coins fuse into ONE marker (the sure foreg...
1,2,0.40,9.6,7,7,2,1,1.049,Only 7 markers for 24 coins: chains of touchin...
2,3,0.60,14.4,24,24,0,0,0.042,"24 markers / 24 regions, area spread 0.042: ev..."
3,4,0.80,19.2,24,24,0,0,0.037,"24 markers / 24 regions, area spread 0.037: st..."
4,5,0.95,22.8,24,24,3,3,0.510,24 markers but markers are tiny cores: 3 overs...


In [22]:
bgr9 = load_color('task9.jpg')
pix = bgr9.reshape((-1, 3)).astype(np.float32)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
ATTEMPTS = 5
cv2.setRNGSeed(0)

def mse_psnr(a, b):
    mse = float(np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2))
    return mse, (10 * np.log10(255 ** 2 / mse) if mse > 0 else float('inf'))

seg9, rows, info = {}, [], {}
for K in [2, 4, 6]:
    compact, labels, centers = cv2.kmeans(pix, K, None, criteria, ATTEMPTS, cv2.KMEANS_PP_CENTERS)
    centers_u8 = np.uint8(centers)
    seg = centers_u8[labels.flatten()].reshape(bgr9.shape)
    seg9[K] = seg
    mse, psnr = mse_psnr(bgr9, seg)
    pct = np.bincount(labels.flatten(), minlength=K) / labels.size * 100
    order = np.argsort(-pct)
    info[K] = [('#%02x%02x%02x' % tuple(int(v) for v in centers_u8[i][::-1]), round(float(pct[i]), 1)) for i in order]
    rows.append([K, len(np.unique(seg.reshape(-1, 3), axis=0)), round(mse, 1), round(psnr, 2)])
    save(f'task9_kmeans_K{K}.png', seg)
orig_colors = len(np.unique(bgr9.reshape(-1, 3), axis=0))
tab9 = pd.DataFrame(rows, columns=['K (clusters)', 'distinct colours after', 'MSE vs original', 'PSNR (dB)'])
tab9['distinct colours before'] = orig_colors
show_row([('Original', bgr9)] + [(f'K = {K}', seg9[K]) for K in [2, 4, 6]], save_as='task9_kmeans_comparison.png', size=4.4)
for K in info: print(f'K={K}: cluster colours (hex, % of pixels):', info[K])
tab9

K=2: cluster colours (hex, % of pixels): [('#0a0a11', 87.8), ('#999090', 12.2)]
K=4: cluster colours (hex, % of pixels): [('#030205', 67.5), ('#19233a', 18.5), ('#74615f', 9.0), ('#cdc8c9', 4.9)]
K=6: cluster colours (hex, % of pixels): [('#030203', 64.7), ('#0d1b33', 16.6), ('#3d4c61', 6.6), ('#928c8f', 4.9), ('#7b381d', 4.0), ('#e0dbda', 3.2)]


,K (clusters),distinct colours after,MSE vs original,PSNR (dB),distinct colours before
0,2,2,877.7,18.70,52354
1,4,4,373.9,22.40,52354
2,6,6,240.9,24.31,52354


In [23]:
bgr10 = load_color('task10.jpg')
gray10 = cv2.cvtColor(bgr10, cv2.COLOR_BGR2GRAY)
hsv10 = cv2.cvtColor(bgr10, cv2.COLOR_BGR2HSV)
h10, w10 = gray10.shape
print("image:", bgr10.shape)
# approximate hand-placed box around the dog, only used to measure how much of each mask falls on the dog (a rough reference, not a ground truth)
DOG_BOX = (72, 188, 172, 278)    # x0, y0, x1, y1
x0, y0, x1, y1 = DOG_BOX
box_vis = bgr10.copy(); cv2.rectangle(box_vis, (x0, y0), (x1, y1), (0, 255, 0), 2)
show_row([('Image with reference box on the dog', box_vis)], size=4)

image: (350, 233, 3)


In [24]:
# ---- Method 1: Otsu on the blurred gray image (bright pixels = foreground)
blur10 = cv2.GaussianBlur(gray10, (5, 5), 0)
t_o, m1 = cv2.threshold(blur10, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("Method 1  Otsu threshold:", t_o)

# ---- Method 2: HSV colour thresholding for 'white' (low saturation, high value)
HSV_LO, HSV_HI = (0, 0, 165), (180, 70, 255)
m2 = cv2.inRange(hsv10, np.array(HSV_LO, np.uint8), np.array(HSV_HI, np.uint8))
m2 = cv2.morphologyEx(m2, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
print("Method 2  HSV range:", HSV_LO, HSV_HI)

# ---- Method 3: K-Means (K=4) on BGR pixels; the brightest cluster is the foreground
pix10 = cv2.GaussianBlur(bgr10, (5, 5), 0).reshape(-1, 3).astype(np.float32)
cv2.setRNGSeed(0)
_, lab10, cen10 = cv2.kmeans(pix10, 4, None, (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0), 5, cv2.KMEANS_PP_CENTERS)
bright = int(np.argmax(cen10.sum(axis=1)))
m3 = (lab10.reshape(h10, w10) == bright).astype(np.uint8) * 255
km_vis = np.uint8(cen10)[lab10.flatten()].reshape(bgr10.shape)
print("Method 3  K-Means centres (BGR):", np.uint8(cen10).tolist(), "| brightest cluster =", bright)

def score(mask):
    fg = mask > 0
    inside = fg[y0:y1, x0:x1].sum(); total = fg.sum()
    return dict(fg_pct=round(100 * fg.mean(), 2), inside_box_pct_of_mask=round(100 * inside / max(total, 1), 1),
                dog_box_covered_pct=round(100 * inside / ((y1 - y0) * (x1 - x0)), 1))
tab10 = pd.DataFrame({'Method 1: Otsu': score(m1), 'Method 2: HSV colour': score(m2), 'Method 3: K-Means K=4': score(m3)}).T
tab10

Method 1  Otsu threshold: 111.0
Method 2  HSV range: (0, 0, 165) (180, 70, 255)
Method 3  K-Means centres (BGR): [[69, 82, 113], [176, 174, 196], [54, 56, 75], [108, 119, 158]] | brightest cluster = 1


,fg_pct,inside_box_pct_of_mask,dog_box_covered_pct
Method 1: Otsu,23.05,31.1,64.9
Method 2: HSV colour,6.79,59.6,36.7
Method 3: K-Means K=4,8.54,51.3,39.7


In [25]:
show_row([('Original', bgr10), (f'Method 1: Otsu (T={t_o:.0f})', m1), ('Method 2: HSV colour (S<=70, V>=165)', m2), ('Method 3: K-Means K=4 (brightest cluster)', m3)],
         save_as='task10_final_comparison.png', size=4.2)
save('task10_otsu.png', m1); save('task10_hsv.png', m2); save('task10_kmeans_mask.png', m3); save('task10_kmeans_segmented.png', km_vis)
tab10.to_csv(os.path.join(OUT, 'task10_scores.csv'))

In [26]:
for f in sorted(os.listdir(OUT)): print(f)

task10_final_comparison.png
task10_hsv.png
task10_kmeans_mask.png
task10_kmeans_segmented.png
task10_otsu.png
task10_scores.csv
task1_best_global_mask.png
task1_comparison.png
task1_final_adaptive_mask.png
task2_best_mask.png
task2_final_comparison.png
task2_gaussian_grid.png
task2_mean_grid.png
task2_parameter_table.csv
task3_original_hist_otsu.png
task3_otsu_mask.png
task3_threshold_comparison.csv
task3_variants.png
task4_comparison.png
task4_final_extraction.png
task4_final_mask.png
task4_restrictive_mask.png
task5_canny_comparison.png
task5_final_edges.png
task5_fixed_low_sweep.png
task5_strong_weak.png
task6_final_mask.png
task6_final_mask_view.png
task6_parameter_table.csv
task6_region_growing.png
task7_final_watershed.png
task7_watershed_pipeline.png
task8_best_watershed.png
task8_table.csv
task8_tuning.png
task9_kmeans_K2.png
task9_kmeans_K4.png
task9_kmeans_K6.png
task9_kmeans_comparison.png
